# 04 — Agentic RAG

No Agentic RAG, o LLM toma decisoes autonomas sobre o processo de recuperacao.
Em vez de seguir um pipeline fixo, o agente decide:
- Se precisa buscar informacoes
- O que buscar
- Se o resultado e suficiente
- Se precisa reformular a query

## Diagrama

```
Query
  ↓
[AGENTE] - Precisa buscar?
  ├── Nao → Responder diretamente
  └── Sim → Formular query de busca
               ↓
           [RETRIEVER] - Buscar documentos
               ↓
           [AGENTE] - Resultado e suficiente?
               ├── Sim → Gerar resposta final
               └── Nao → Reformular query → loop
```

**Prerequisito:** Ollama com llama3.2

In [ ]:
import sys
sys.path.insert(0, '..')

from src.rag.agentic import AgenticRAG
from pathlib import Path
from src.utils.chunking import recursive_chunk
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
import json

model = SentenceTransformer('all-MiniLM-L6-v2')
client = QdrantClient(host='localhost', port=6333)

# Indexar se necessario
COLLECTION = 'agentic_rag'
if COLLECTION not in [c.name for c in client.get_collections().collections]:
    docs_dir = Path('../data/sample_docs')
    chunks = []
    for p in docs_dir.glob('*.md'):
        for c in recursive_chunk(p.read_text(encoding='utf-8'), chunk_size=400, overlap=50):
            chunks.append(c)
    
    embs = model.encode([c.text for c in chunks], normalize_embeddings=True, show_progress_bar=True)
    if client.collection_exists(COLLECTION):
        client.delete_collection(COLLECTION)
    client.create_collection(COLLECTION, vectors_config=VectorParams(size=384, distance=Distance.COSINE))
    points = [PointStruct(id=i, vector=embs[i].tolist(), payload={'text': chunks[i].text}) for i in range(len(chunks))]
    client.upsert(COLLECTION, points=points)
    print(f'{len(chunks)} chunks indexados')

agent = AgenticRAG(
    collection_name=COLLECTION,
    max_iterations=3,
    top_k=5,
)
print('AgenticRAG pronto!')

## 4.1 Loop Agente: Observar o processo de decisao

In [ ]:
import time

# Pergunta simples — agente deve decidir rapidamente
query_simples = 'O que e embeddings de texto?'

print(f'Query: {query_simples}')
print('Acompanhe as decisoes do agente:')  
print('='*60)

t0 = time.time()
resultado = agent.query(query_simples, verbose=True)
print(f'\nTempo total: {time.time()-t0:.2f}s')
print(f'Iteracoes: {resultado["iterations"]}')
print(f'Terminado por: {resultado["terminated"]}')

In [ ]:
# Pergunta complexa — agente pode iterar mais
query_complexa = ('Dado que temos restricao de memoria de 2GB para vetores, '
                  'qual combinacao de dimensao e tipo de float e mais adequada '
                  'para indexar 1 milhao de documentos mantendo alta qualidade?')

print(f'Query complexa: {query_complexa}')
print('='*60)

t0 = time.time()
resultado = agent.query(query_complexa, verbose=True)
print(f'\nTempo total: {time.time()-t0:.2f}s')
print(f'Iteracoes: {resultado["iterations"]}')
print(f'\nResposta Final:')
print(resultado['answer'])

## 4.2 Agentic RAG com LangGraph

LangGraph permite construir agentes RAG mais sofisticados com grafos de estado.

In [ ]:
# Demonstracao conceitual de um grafo LangGraph
# (requer langgraph instalado)

print('Estrutura de um grafo LangGraph para Agentic RAG:')
print()
print('Nos do grafo:')
print('  1. route_query    → Decide: resposta direta ou busca?')
print('  2. retrieve       → Busca no Qdrant')
print('  3. grade_docs     → Avalia relevancia dos docs')
print('  4. rewrite_query  → Reformula query se docs insuficientes')
print('  5. generate       → Gera resposta final')
print()
print('Arestas:')
print('  route_query → [retrieve | generate_direct]')
print('  retrieve → grade_docs')
print('  grade_docs → [generate | rewrite_query]')
print('  rewrite_query → retrieve (loop)')
print('  generate → END')

# Diagrama ASCII do grafo
print()
print('Diagrama do grafo:')
print('''
        [START]
           |
      [route_query]
      /          \\
 [retrieve]   [generate_direct]
     |               |
 [grade_docs]       END
  /        \\
[generate] [rewrite_query]
   |            |
  END      [retrieve]  ← loop
''')

In [ ]:
# Implementacao minimalista com LangGraph
try:
    from langgraph.graph import StateGraph, END
    from typing import TypedDict, List, Optional
    import ollama as ol
    
    class RAGState(TypedDict):
        question: str
        documents: List[str]
        generation: Optional[str]
        iterations: int
    
    def retrieve_node(state: RAGState) -> RAGState:
        q_vec = model.encode(state['question'], normalize_embeddings=True)
        results = client.query_points(COLLECTION, query=q_vec.tolist(), limit=5, with_payload=True).points
        docs = [r.payload.get('text', '') for r in results]
        return {'documents': docs, 'iterations': state.get('iterations', 0) + 1}
    
    def grade_docs_node(state: RAGState) -> RAGState:
        return state  # simplificado: sempre suficiente
    
    def generate_node(state: RAGState) -> RAGState:
        if not LLM:
            return {'generation': '[LLM offline]'}
        context = '\n'.join(state['documents'][:3])
        prompt = f'Contexto: {context}\n\nPergunta: {state["question"]}\n\nResposta:'
        response = ol.chat(model=LLM, messages=[{'role': 'user', 'content': prompt}])
        return {'generation': response['message']['content']}
    
    def should_continue(state: RAGState) -> str:
        if state.get('iterations', 0) >= 2:
            return 'generate'
        return 'grade'
    
    # Construir grafo
    workflow = StateGraph(RAGState)
    workflow.add_node('retrieve', retrieve_node)
    workflow.add_node('grade', grade_docs_node)
    workflow.add_node('generate', generate_node)
    
    workflow.set_entry_point('retrieve')
    workflow.add_edge('retrieve', 'grade')
    workflow.add_conditional_edges('grade', should_continue, {'generate': 'generate', 'grade': 'retrieve'})
    workflow.add_edge('generate', END)
    
    app = workflow.compile()
    
    # Executar
    query = 'O que e HNSW e quais sao seus parametros?'
    result = app.invoke({'question': query, 'documents': [], 'iterations': 0})
    
    print(f'LangGraph Agentic RAG')
    print(f'Query: {query}')
    print(f'Iteracoes: {result["iterations"]}')
    print(f'Resposta: {result["generation"][:300]}')
    
except ImportError:
    print('LangGraph nao instalado. Execute: uv add langgraph')
except Exception as e:
    print(f'Erro: {e}')

## Quando usar Agentic RAG?

| Cenario | Arquitetura Recomendada |
|---------|------------------------|
| Queries simples e claras | Naive RAG ou Advanced RAG |
| Queries complexas multi-step | **Agentic RAG** |
| Q&A sobre multiplos documentos | **Agentic RAG** |
| Latencia critica (<1s) | Naive RAG |
| Qualidade maxima, latencia tolerada | **Agentic RAG** |

## Proximo
- [05 — GraphRAG](05_graphrag.html)